## Step 6 — assign admin id to blocks dataset
**# of cells in notebook:** 1

**Purpose:** We want to identify a given block as belonging to a given admin unit. These are the units from the geoboundaries layer from the previous step.  

**Input:**

- a geodatabase with: `zones_1`, `geoboundaries_selection`
- a geodatabase with: the blocks layer from step 5. 

**Output:** `zones_1` with column `geoboundaries`

**Main logic:**

1. convert the block polygons to inside-constrained centroid points, so each block is represented by one point that is guaranteed to fall inside the block polygon.
2. spatially join the geoboundary attributes to the block centroid points, so each block centroid receives the `OBJECTID` of the geoboundary polygon it falls within.
3. build a lookup table from block `OBJECTID` to geoboundary `OBJECTID`, using the centroid spatial join result. Blocks whose centroids do not intersect a geoboundary are assigned NULL.
4. write the matched geoboundary `OBJECTID` back to the original blocks layer in the `geoboundaries` field. Existing values in that field are overwritten, and temporary scratch outputs are deleted at the end.

In [ ]:
import arcpy
import os
import time
import traceback

# ============================================================
# USER INPUTS
# ============================================================

geoboundaries = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb\geoboundaries_selection"

blocks = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb\juba_blocks_20260415_small_utm36n"

# New / updated field to create in blocks
out_field = "geoboundaries"


# ============================================================
# SETTINGS
# ============================================================

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

scratch_gdb = arcpy.env.scratchGDB

geoboundaries_tmp = os.path.join(scratch_gdb, "tmp_geoboundaries_with_oid")
block_centroids = os.path.join(scratch_gdb, "tmp_block_centroids_inside")
centroids_sj = os.path.join(scratch_gdb, "tmp_block_centroids_geoboundaries_sj")

geoboundaries_oid_tmp_field = "gb_oid_tmp"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def msg(text):
    print(text)
    arcpy.AddMessage(text)


def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)


def check_exists(path, label):
    if not arcpy.Exists(path):
        raise FileNotFoundError(f"{label} does not exist:\n{path}")


def get_field_name(table, field_name):
    for f in arcpy.ListFields(table):
        if f.name.lower() == field_name.lower():
            return f.name
    return None


def field_exists(table, field_name):
    return get_field_name(table, field_name) is not None


def print_fields(table, label):
    msg(f"\nFields in {label}:")
    for f in arcpy.ListFields(table):
        msg(f"  {f.name} | {f.type}")


def describe_basic(path, label):
    desc = arcpy.Describe(path)
    count = int(arcpy.management.GetCount(path)[0])

    msg(f"\n{label}")
    msg(f"  Path: {path}")
    msg(f"  Shape type: {desc.shapeType}")
    msg(f"  Feature count: {count}")
    msg(f"  OID field: {desc.OIDFieldName}")
    msg(f"  Spatial reference: {desc.spatialReference.name}")

    return desc, count


# ============================================================
# MAIN SCRIPT
# ============================================================

try:
    t0 = time.time()

    msg("Starting geoboundaries OBJECTID assignment to blocks...")

    # --------------------------------------------------------
    # Check inputs
    # --------------------------------------------------------

    check_exists(geoboundaries, "geoboundaries_selection")
    check_exists(blocks, "blocks")

    geoboundaries_desc, geoboundaries_count = describe_basic(
        geoboundaries,
        "Input geoboundaries_selection"
    )

    blocks_desc, blocks_count = describe_basic(
        blocks,
        "Input blocks"
    )

    if geoboundaries_desc.shapeType != "Polygon":
        raise ValueError(
            f"geoboundaries_selection must be polygon, but it is: {geoboundaries_desc.shapeType}"
        )

    if blocks_desc.shapeType != "Polygon":
        raise ValueError(
            f"blocks must be polygon, but it is: {blocks_desc.shapeType}"
        )

    geoboundaries_oid_field = geoboundaries_desc.OIDFieldName
    blocks_oid_field = blocks_desc.OIDFieldName

    if geoboundaries_desc.spatialReference.name != blocks_desc.spatialReference.name:
        msg("\nWARNING: geoboundaries and blocks have different spatial references.")
        msg("For best results, project them to the same CRS before running this workflow.")
        msg(f"  geoboundaries CRS: {geoboundaries_desc.spatialReference.name}")
        msg(f"  blocks CRS:        {blocks_desc.spatialReference.name}")

    # --------------------------------------------------------
    # Clean temporary outputs
    # --------------------------------------------------------

    msg("\nCleaning temporary outputs...")

    delete_if_exists(geoboundaries_tmp)
    delete_if_exists(block_centroids)
    delete_if_exists(centroids_sj)

    # --------------------------------------------------------
    # Copy geoboundaries and preserve original OBJECTID
    # --------------------------------------------------------

    msg("\nCreating temporary geoboundaries layer with preserved OBJECTID...")

    arcpy.management.CopyFeatures(
        in_features=geoboundaries,
        out_feature_class=geoboundaries_tmp
    )

    if field_exists(geoboundaries_tmp, geoboundaries_oid_tmp_field):
        arcpy.management.DeleteField(geoboundaries_tmp, geoboundaries_oid_tmp_field)

    arcpy.management.AddField(
        in_table=geoboundaries_tmp,
        field_name=geoboundaries_oid_tmp_field,
        field_type="LONG"
    )

    arcpy.management.CalculateField(
        in_table=geoboundaries_tmp,
        field=geoboundaries_oid_tmp_field,
        expression=f"!{geoboundaries_oid_field}!",
        expression_type="PYTHON3"
    )

    msg(f"  Temporary geoboundaries created: {geoboundaries_tmp}")
    msg(f"  Preserved geoboundaries OBJECTID in field: {geoboundaries_oid_tmp_field}")

    # --------------------------------------------------------
    # Create inside block centroid points
    # --------------------------------------------------------

    msg("\nCreating inside-constrained block centroids...")

    arcpy.management.FeatureToPoint(
        in_features=blocks,
        out_feature_class=block_centroids,
        point_location="INSIDE"
    )

    centroid_count = int(arcpy.management.GetCount(block_centroids)[0])
    msg(f"  Created block centroids: {block_centroids}")
    msg(f"  Centroid count: {centroid_count}")

    if not field_exists(block_centroids, "ORIG_FID"):
        print_fields(block_centroids, "block centroids")
        raise RuntimeError("Expected ORIG_FID field was not created on block centroids.")

    # --------------------------------------------------------
    # Spatial join centroids to geoboundaries
    # --------------------------------------------------------

    msg("\nSpatial joining block centroids to geoboundaries polygons...")

    arcpy.analysis.SpatialJoin(
        target_features=block_centroids,
        join_features=geoboundaries_tmp,
        out_feature_class=centroids_sj,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_ALL",
        match_option="INTERSECT"
    )

    sj_count = int(arcpy.management.GetCount(centroids_sj)[0])
    msg(f"  Spatial join output: {centroids_sj}")
    msg(f"  Spatial join count: {sj_count}")

    if not field_exists(centroids_sj, "ORIG_FID"):
        print_fields(centroids_sj, "spatial join output")
        raise RuntimeError("Spatial join output is missing ORIG_FID.")

    if not field_exists(centroids_sj, geoboundaries_oid_tmp_field):
        print_fields(centroids_sj, "spatial join output")
        raise RuntimeError(
            f"Spatial join output is missing {geoboundaries_oid_tmp_field}."
        )

    # --------------------------------------------------------
    # Build dictionary: block OBJECTID -> geoboundaries OBJECTID
    # --------------------------------------------------------

    msg("\nBuilding block-to-geoboundaries lookup...")

    block_to_geoboundary = {}
    unmatched_centroids = 0

    with arcpy.da.SearchCursor(
        centroids_sj,
        ["ORIG_FID", geoboundaries_oid_tmp_field]
    ) as cursor:
        for block_oid, gb_oid in cursor:
            if gb_oid is None:
                unmatched_centroids += 1
                block_to_geoboundary[int(block_oid)] = None
            else:
                block_to_geoboundary[int(block_oid)] = int(gb_oid)

    matched = sum(v is not None for v in block_to_geoboundary.values())

    msg(f"  Block centroids with matched geoboundary: {matched}")
    msg(f"  Block centroids without matched geoboundary: {unmatched_centroids}")

    # --------------------------------------------------------
    # Add output field to blocks if needed
    # --------------------------------------------------------

    msg(f"\nPreparing output field in blocks: {out_field}")

    if not field_exists(blocks, out_field):
        arcpy.management.AddField(
            in_table=blocks,
            field_name=out_field,
            field_type="LONG",
            field_is_nullable="NULLABLE"
        )
        msg(f"  Added field: {out_field}")
    else:
        msg(f"  Field already exists: {out_field}")
        msg("  Existing values will be overwritten.")

    out_field_actual = get_field_name(blocks, out_field)

    # --------------------------------------------------------
    # Write geoboundaries OBJECTID values back to blocks
    # --------------------------------------------------------

    msg("\nWriting geoboundaries OBJECTID values back to blocks...")

    updated = 0
    no_match = 0
    not_in_lookup = 0

    with arcpy.da.UpdateCursor(blocks, [blocks_oid_field, out_field_actual]) as cursor:
        for block_oid, current_value in cursor:
            block_oid_int = int(block_oid)

            if block_oid_int not in block_to_geoboundary:
                cursor.updateRow([block_oid, None])
                not_in_lookup += 1
                continue

            gb_oid = block_to_geoboundary[block_oid_int]

            if gb_oid is None:
                cursor.updateRow([block_oid, None])
                no_match += 1
            else:
                cursor.updateRow([block_oid, gb_oid])
                updated += 1

    msg(f"  Updated blocks with geoboundaries OBJECTID: {updated}")
    msg(f"  Blocks with no matched geoboundary: {no_match}")
    msg(f"  Blocks missing from centroid lookup: {not_in_lookup}")

    # --------------------------------------------------------
    # Clean temporary outputs
    # --------------------------------------------------------

    msg("\nCleaning temporary outputs...")

    delete_if_exists(geoboundaries_tmp)
    delete_if_exists(block_centroids)
    delete_if_exists(centroids_sj)

    elapsed = round((time.time() - t0) / 60, 2)
    msg(f"\nDone. Elapsed time: {elapsed} minutes")

except Exception as e:
    msg("\nSCRIPT FAILED.")
    msg(str(e))
    msg(traceback.format_exc())
    raise